# NQ Research Platform

Research NQ futures setups around the **9:30 AM New York open**, entirely in your browser.

### How to use this page
1. Click **Runtime → Run all** in the menu at the top. (Or press the ▶ button on each box, top to bottom.)
2. If Google shows **"Warning: This notebook was not authored by Google"**, click **Run anyway**.
3. Wait for each box to finish. Results appear underneath it.

To change a setting, edit it in the **Settings** box, then click **Runtime → Run all** again.

Each grey box has hidden code. Double-click a box's title to see the code. You don't need to.

In [ ]:
#@title ⚙️ Settings
#@markdown **Where the price data comes from**
DATA_SOURCE = "Yahoo (free, last 60 days)" #@param ["Yahoo (free, last 60 days)", "Upload a TradingView export"]

#@markdown **The open** (New York time). 09:30 = stock market cash open, 08:30 = economic data releases.
OPEN_TIME = "09:30" #@param ["09:30", "08:30"]

#@markdown **Opening range length**, in minutes after the open
OPENING_RANGE_MINUTES = 15 #@param {type:"integer"}

#@markdown **How long to keep watching price** after the open, in minutes
WATCH_MINUTES = 120 #@param {type:"integer"}

#@markdown **Day to draw on the chart** in step 3, written like 2026-10-06. Leave empty for the most recent day.
CHART_DAY = "" #@param {type:"string"}

SYMBOL = "NQ=F"          # Yahoo's name for front-month NQ futures
TIMEZONE = "America/New_York"
print("Settings saved.")

In [ ]:
#@title 1️⃣ Load price data
import io

import pandas as pd

try:
    import yfinance as yf
except ImportError:
    # Colab normally has this already; install it if it's missing.
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=True)
    import yfinance as yf


def read_tradingview(file_bytes):
    """Turn a TradingView "Export chart data" CSV into our candle table."""
    raw = pd.read_csv(io.BytesIO(file_bytes))
    raw.columns = [column.strip().lower() for column in raw.columns]

    missing = [c for c in ["time", "open", "high", "low", "close"] if c not in raw.columns]
    if missing:
        raise SystemExit(f"This doesn't look like a TradingView export. Missing columns: {missing}")

    # TradingView exports time as "UNIX timestamp" (a big number) or "ISO time" (readable text).
    if pd.api.types.is_numeric_dtype(raw["time"]):
        times = pd.to_datetime(raw["time"], unit="s", utc=True)
    else:
        times = pd.to_datetime(raw["time"], utc=True)

    table = pd.DataFrame({
        "Open": raw["open"].values,
        "High": raw["high"].values,
        "Low": raw["low"].values,
        "Close": raw["close"].values,
        "Volume": raw["volume"].values if "volume" in raw.columns else 0,
    }, index=times)
    return table


if DATA_SOURCE.startswith("Yahoo"):
    print("Downloading 60 days of 5-minute NQ candles from Yahoo...")
    candles = yf.download(SYMBOL, period="60d", interval="5m", progress=False, auto_adjust=False)
    if candles.empty:
        raise SystemExit("No data came back from Yahoo. Wait a minute and run this box again.")
    # Yahoo gives two-level column names ("Close" / "NQ=F"). Keep the first level only.
    candles.columns = candles.columns.get_level_values(0)
    candles = candles[["Open", "High", "Low", "Close", "Volume"]]
else:
    from google.colab import files
    print("Click 'Choose Files' below and pick your TradingView CSV export.")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file was chosen. Run this box again.")
    file_name, file_bytes = next(iter(uploaded.items()))
    print(f"Reading {file_name}...")
    candles = read_tradingview(file_bytes)

candles.index = candles.index.tz_convert(TIMEZONE)
candles.index.name = "Time"
candles = candles.sort_index()

minutes = candles.index.to_series().diff().dropna().dt.total_seconds().median() / 60
print(f"Loaded {len(candles):,} candles of about {minutes:g} minutes each")
print(f"From {candles.index[0]:%Y-%m-%d %H:%M} to {candles.index[-1]:%Y-%m-%d %H:%M} New York time")
print(f"Trading days: {len(set(candles.index.date))}")

In [ ]:
#@title 2️⃣ Opening range study
#@markdown For every day: the high and low of the opening range, which side broke first,
#@markdown and how far price travelled past the range afterwards.
from datetime import timedelta

from IPython.display import display

open_hour, open_minute = map(int, OPEN_TIME.split(":"))
results = []

for day, day_candles in candles.groupby(candles.index.date):
    open_time = pd.Timestamp(day, tz=TIMEZONE).replace(hour=open_hour, minute=open_minute)
    range_end = open_time + timedelta(minutes=OPENING_RANGE_MINUTES)
    watch_end = open_time + timedelta(minutes=WATCH_MINUTES)

    range_candles = day_candles[(day_candles.index >= open_time) & (day_candles.index < range_end)]
    after_candles = day_candles[(day_candles.index >= range_end) & (day_candles.index < watch_end)]

    # Skip weekends, holidays, and days with missing data.
    if range_candles.empty or after_candles.empty:
        continue

    range_high = range_candles["High"].max()
    range_low = range_candles["Low"].min()
    broke_up = after_candles[after_candles["High"] > range_high]
    broke_down = after_candles[after_candles["Low"] < range_low]

    if broke_up.empty and broke_down.empty:
        first_break = "none"
    elif broke_down.empty or (not broke_up.empty and broke_up.index[0] < broke_down.index[0]):
        first_break = "up"
    elif broke_up.empty or broke_down.index[0] < broke_up.index[0]:
        first_break = "down"
    else:
        # Both sides broke inside the same candle - we can't tell which came first.
        first_break = "same candle"

    results.append({
        "date": day,
        "range_high": range_high,
        "range_low": range_low,
        "range_size": range_high - range_low,
        "first_break": first_break,
        "broke_up": not broke_up.empty,
        "broke_down": not broke_down.empty,
        "max_points_above": max(after_candles["High"].max() - range_high, 0),
        "max_points_below": max(range_low - after_candles["Low"].min(), 0),
    })

if not results:
    raise SystemExit(f"No days had data at {OPEN_TIME}. Check the data loaded in step 1.")

table = pd.DataFrame(results)

print(f"Opening range: first {OPENING_RANGE_MINUTES} minutes after {OPEN_TIME} New York")
print(f"Watching price until {WATCH_MINUTES} minutes after the open")
print()
print("=========== SUMMARY ===========")
print(f"Days studied:                  {len(table)}")
print(f"Average range size:            {table['range_size'].mean():.1f} points")
print(f"Days that broke up first:      {(table['first_break'] == 'up').sum()}")
print(f"Days that broke down first:    {(table['first_break'] == 'down').sum()}")
print(f"Days that broke BOTH sides:    {(table['broke_up'] & table['broke_down']).sum()}")
print(f"Days that never broke out:     {(table['first_break'] == 'none').sum()}")
print(f"Average move above the range:  {table['max_points_above'].mean():.1f} points")
print(f"Average move below the range:  {table['max_points_below'].mean():.1f} points")
print()
display(table.round(2))

In [ ]:
#@title 3️⃣ Chart one day
#@markdown Candles from 1 hour before the open to the end of the watch window.
#@markdown Green and red lines = opening range high and low. Pick the day in **Settings**.
import plotly.graph_objects as go

chart_day = pd.Timestamp(CHART_DAY).date() if CHART_DAY.strip() else table["date"].iloc[-1]
day_row = table[table["date"] == chart_day]
if day_row.empty:
    raise SystemExit(f"{chart_day} isn't in the study. Pick a date from the table in step 2.")
day_row = day_row.iloc[0]

open_time = pd.Timestamp(chart_day, tz=TIMEZONE).replace(hour=open_hour, minute=open_minute)
window = candles[(candles.index >= open_time - timedelta(hours=1))
                 & (candles.index < open_time + timedelta(minutes=WATCH_MINUTES))]

figure = go.Figure(go.Candlestick(
    x=window.index, open=window["Open"], high=window["High"],
    low=window["Low"], close=window["Close"], name="NQ",
))
figure.add_hline(y=day_row["range_high"], line_color="green", annotation_text="Range high")
figure.add_hline(y=day_row["range_low"], line_color="red", annotation_text="Range low")
figure.add_vline(x=open_time, line_dash="dot", line_color="gray")
figure.update_layout(
    title=f"NQ {chart_day}  -  first break: {day_row['first_break']}",
    xaxis_rangeslider_visible=False, height=550,
)
figure.show()

In [ ]:
#@title 💾 Download the results table
#@markdown Saves the step 2 table as a CSV file you can open in Excel, Numbers or Google Sheets.
table.to_csv("opening_range.csv", index=False)
try:
    from google.colab import files
    files.download("opening_range.csv")
    print("Download started. Check your browser's downloads.")
except ImportError:
    print("Saved opening_range.csv")